In [40]:
import re
from pathlib import Path
from pprint import pprint

import torch
from sympy.parsing import sympy_parser as spp
from sympy.core.sympify import SympifyError
from sympy import simplify
from sympy.polys.polyerrors import PolynomialError
from tokenize import TokenError

from reasoning_from_scratch.ch02 import (
    get_device, 
    generate_text_basic_stream_cache
)
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)
from IPython.display import Latex, Math, display 

## 3.2 Loading a pretrained model to generate text

In [2]:
def load_model_and_tokenizer(
    which_model, 
    device, 
    use_compile, 
    local_dir='qwen3'
):
    if which_model == "base":
        download_qwen3_small(
            kind="base", 
            tokenizer_only=False, 
            out_dir=local_dir
        )

        tokenizer_path = Path(local_dir) / "tokenizer-base.json"
        model_path = Path(local_dir) / "qwen3-0.6B-base.pth"
        tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

    elif which_model == "reasoning":

        download_qwen3_small(
            kind="reasoning", 
            tokenizer_only=False, 
            out_dir=local_dir
        )

        tokenizer_path = Path(local_dir) / "tokenizer-reasoning.json"
        model_path = Path(local_dir) / "qwen3-0.6B-reasoning.pth"
        tokenizer = Qwen3Tokenizer(
            tokenizer_file_path=tokenizer_path,
            apply_chat_template=True,
            add_generation_prompt=True,
            add_thinking=True,
        )
    
    else:
        raise ValueError(f"Invalid choice: which_model={which_model}")

    model = Qwen3Model(QWEN_CONFIG_06_B)
    model.load_state_dict(torch.load(model_path))

    model.to(device)

    if use_compile:
        torch._dynamo.config.allow_unspec_int_on_nn_module = True
        model = torch.compile(model)

    return model, tokenizer

WHICH_MODEL = "base"
device = get_device()


model, tokenizer = load_model_and_tokenizer(
    which_model=WHICH_MODEL,
    device=device,
    use_compile=False
)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [7]:
prompt = (
    r"If $a+b=3$ and $ab=\tfrac{13}{6}, "
    r"what is the value of $a^2+b^2$?"
)

input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
).unsqueeze(0)

all_token_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=2048,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0)
    decoded_id = tokenizer.decode(token_id.tolist())
    print(decoded_id, end="", flush=True)
    all_token_ids.append(token_id)

all_tokens = tokenizer.decode(all_token_ids)
display(Latex(all_tokens))

 To find the value of \( a^2 + b^2 \) given that \( a + b = 3 \) and \( ab = \frac{13}{6} \), we can use the following algebraic identity:

\[
a^2 + b^2 = (a + b)^2 - 2ab
\]

**Step 1:** Substitute the given values into the equation.

\[
a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
\]

**Step 2:** Calculate \( (3)^2 \).

\[
(3)^2 = 9
\]

**Step 3:** Calculate \( 2 \times \frac{13}{6} \).

\[
2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3}
\]

**Step 4:** Subtract the second result from the first.

\[
a^2 + b^2 = 9 - \frac{13}{3}
\]

**Step 5:** Convert 9 to a fraction with a denominator of 3 to perform the subtraction.

\[
9 = \frac{27}{3}
\]

\[
a^2 + b^2 = \frac{27}{3} - \frac{13}{3} = \frac{14}{3}
\]

**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]

<IPython.core.display.Latex object>

## 3.3 Implementing a wrapper for easier text generation

In [8]:
def generate_text_stream_concat(
    model, tokenizer, prompt, device, max_new_tokens,
    verbose=False,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt), device=device  
    ).unsqueeze(0)                           

    generated_ids = []
    for token in generate_text_basic_stream_cache(
        model=model,                                
        token_ids=input_ids,                        
        max_new_tokens=max_new_tokens,              
        eos_token_id=tokenizer.eos_token_id,        
    ):                                              
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id.item())

        if verbose:
            print(
                tokenizer.decode(next_token_id.tolist()),
                end="",
                flush=True
            )

    return tokenizer.decode(generated_ids)

generated_text = generate_text_stream_concat(
    model, tokenizer, prompt, device,
    max_new_tokens=2048,
    verbose=True
)

 To find the value of \( a^2 + b^2 \) given that \( a + b = 3 \) and \( ab = \frac{13}{6} \), we can use the following algebraic identity:

\[
a^2 + b^2 = (a + b)^2 - 2ab
\]

**Step 1:** Substitute the given values into the equation.

\[
a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
\]

**Step 2:** Calculate \( (3)^2 \).

\[
(3)^2 = 9
\]

**Step 3:** Calculate \( 2 \times \frac{13}{6} \).

\[
2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3}
\]

**Step 4:** Subtract the second result from the first.

\[
a^2 + b^2 = 9 - \frac{13}{3}
\]

**Step 5:** Convert 9 to a fraction with a denominator of 3 to perform the subtraction.

\[
9 = \frac{27}{3}
\]

\[
a^2 + b^2 = \frac{27}{3} - \frac{13}{3} = \frac{14}{3}
\]

**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]

## 3.4 Extracting the final answer box

In [10]:
model_answer = (
r"""... some explanation...
**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]
""")

In [13]:
def get_last_boxed(text):
    # find the last occurrence of \boxed
    boxed_start_idx = text.rfind(r"\boxed")
    if boxed_start_idx == -1:
        return None

    # get the index right after \boxed
    current_idx = boxed_start_idx + len(r"\boxed")

    # skip any whitespace after \boxed
    while current_idx < len(text) and text[current_idx].isspace():
        current_idx += 1

    # check if the next character is a left brace
    if current_idx >= len(text) or text[current_idx] != "{":
        return None

    # traverse the text and count braces
    current_idx += 1
    brace_depth = 1
    content_start_idx = current_idx
    while current_idx < len(text) and brace_depth > 0:
        char = text[current_idx]
        
        if char == "{":
            brace_depth += 1

        elif char == "}":
            brace_depth -= 1
        
        current_idx += 1

    # if there's not a matching closing brace, return None
    if brace_depth != 0:
        return None

    return text[content_start_idx: current_idx-1]


extracted_answer = get_last_boxed(model_answer)
display(Math(extracted_answer))

<IPython.core.display.Math object>

In [16]:
RE_NUMBER = re.compile(
    r"-?(?:\d+/\d+|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)"
)


def extract_final_candidate(text, fallback='number_then_full'):
    # default return value
    result = ""

    if text:
        boxed = get_last_boxed(text.strip())
        
        if boxed:
            result = boxed.strip().strip("$ ")

        elif fallback in ("number_then_full", "number_only"):
            m = RE_NUMBER.findall(text)
            
            # if we found any numbers, return the last one
            if m:
                result = m[-1]

            # if we didn't find any numbers, and the fallback is 
            # "number_then_full", return the full text
            elif fallback == "number_then_full":
                result = text
    
    return result


print(extract_final_candidate(model_answer))
print(extract_final_candidate(r"\boxed{ 14/3. }"))
print(extract_final_candidate("abc < > 14/3 abc"))

\dfrac{14}{3}
14/3.
14/3


## 3.5 Normalizing the extracted answer

In [17]:
LATEX_FIXES = [
    (r"\\left\s*", ""),
    (r"\\right\s*", ""),
    (r"\\,|\\!|\\;|\\:", ""),
    (r"\\cdot", "*"),
    (r"·|×", "*"),
    (r"\\\^\\circ", ""),
    (r"\\dfrac", r"\\frac"),
    (r"\\tfrac", r"\\frac"),
    (r"°", ""),
]

RE_SPECIAL = re.compile(r"<\|[^>]+?\|>")   # special tokens like <|endoftext|>

SUPERSCRIPT_MAP = {
    "⁰": "0", "¹": "1", "²": "2", "³": "3", "⁴": "4",  
    "⁵": "5", "⁶": "6", "⁷": "7", "⁸": "8", "⁹": "9",   
    "⁺": "+", "⁻": "-", "⁽": "(", "⁾": ")",          
}


def convert_superscripts(s, base=None):
    converted = "".join(
        SUPERSCRIPT_MAP[ch] if ch in SUPERSCRIPT_MAP else ch
        for ch in s
    )
    if base is None:
        return converted
    return f"{base}**{converted}"


def normalize_text(text):
    if not text:
        return ""
    text = RE_SPECIAL.sub("", text).strip()

    # leading letter followed by a period or colon, e.g. "A: " or "B. "
    match = re.match(r"^[A-Za-z]\s*[.:]\s*(.+)$", text)
    if match: 
        text = match.group(1)
    
    # remove LaTeX degree symbols and superscripts
    text = re.sub(r"\^\s*\{\s*\\circ\s*\}", "", text)  
    text = re.sub(r"\^\s*\\circ", "", text)          
    text = text.replace("°", "")

    # remove \text{...} wrappers
    match = re.match(r"^\\text\{(?P<x>.+?)\}$", text)
    if match:
        text = match.group("x")

    # remove \(...\) and \[...\] wrappers
    text = re.sub(r"\\\(|\\\)|\\\[|\\\]", "", text)

    # apply LaTeX fixes
    for pat, rep in LATEX_FIXES:
        text = re.sub(pat, rep, text)

    # convert superscripts to normal text
    text = re.sub(
        r"([0-9A-Za-z\)\]\}])([⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻]+)",
        lambda m: convert_superscripts(m.group(2), base=m.group(1)),
        text,
    )
    text = convert_superscripts(text)

    # remove percent signs and dollar signs, and replace \% with %
    text = text.replace("\\%", "%").replace("$", "").replace("%", "")
    
    # replace \sqrt{...} with sqrt(...) and \sqrt ... with sqrt(...)
    text = re.sub(
        r"\\sqrt\s*\{([^}]*)\}",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )
    text = re.sub(
        r"\\sqrt\s+([^\\\s{}]+)",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )

    # replace \frac{a}{b} with (a)/(b)
    text = re.sub(
        r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )
    text = re.sub(
        r"\\frac\s+([^\s{}]+)\s+([^\s{}]+)",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )

    # handle exponents and mixed numbers
    text = text.replace("^", "**")
    text = re.sub(
        r"(?<=\d)\s+(\d+/\d+)",
        lambda match: "+" + match.group(1),
        text,
    )

    # remove thousands separators
    text = re.sub(
        r"(?<=\d),(?=\d\d\d(\D|$))",
        "",
        text,
    )

    return text.replace("{", "").replace("}", "").strip().lower()


print(normalize_text(extract_final_candidate(model_answer)))
print(normalize_text(r"\text{\[\frac{14}{3}\]}"))

(14)/(3)
(14)/(3)


## 3.6 Verifying mathematical equivalence

In [ ]:
def sympy_parser(expr):
    # avoid crashing on long garbage responses
    if expr is None or len(expr) > 2000:
        return None
    
    try:
        return spp.parse_expr(
            expr,
            transformations=(
                # i.e. handling parenthesis
                *spp.standard_transformations,  
                # allow implicit multiplication symbols
                spp.implicit_multiplication_application,   
            ),

            evaluate=True,  #4
        )
    except (
        SympifyError, 
        SyntaxError, 
        TypeError, 
        AttributeError,
        IndexError, 
        TokenError, 
        ValueError, 
        PolynomialError
    ):
        return None


print(
    sympy_parser(
        normalize_text(
            extract_final_candidate(model_answer)
        )
    )
)
print(sympy_parser("28/6"))

14/3
14/3


In [23]:
def equality_check(expr_gtruth, expr_pred):
    if expr_gtruth == expr_pred:
        return True
    
    gtruth, pred = sympy_parser(expr_gtruth), sympy_parser(expr_pred)

    if gtruth is not None and pred is not None:
        try:
            return simplify(gtruth - pred) == 0
        except (SympifyError, TypeError):
            pass
    
    return False

print(
    equality_check(
        normalize_text("13/4."),
        normalize_text(r"(13)/(4)")
    )
)

True


In [24]:
print(
    equality_check(
        normalize_text("0.5"),
        normalize_text(r"(1)/(2)")
    )
)

True


In [25]:
print(
    equality_check(
        normalize_text("14/3"),
        normalize_text("15/3")
    )
)

False


In [ ]:
print ( 
    equality_check(
        normalize_text("(14/3, 2/3)"),
        normalize_text("(14/3, 4/6)")
    )
)

False


## 3.7 Grading answers

In [27]:
def split_into_parts(text):
    result = [text]

    if text:
        if (
            len(text) >= 2
            and text[0] in "([" and text[-1] in ")]"
            and "," in text[1:-1]
        ):
            items = [p.strip() for p in text[1:-1].split(",")]
            if all(items):
                result = items
    else:
        result = []
    
    return result

split_into_parts(normalize_text(r"( 14/3, 2/3 )"))

['14/3', '2/3']

In [35]:
def grade_answer(pred_text, gt_text): 
    result = False

    if pred_text is not None and gt_text is not None :
        gt_parts = split_into_parts(normalize_text(gt_text))
        pred_parts = split_into_parts(normalize_text(pred_text))

        if (
            gt_parts and pred_parts
            and len(gt_parts) == len(pred_parts)
        ):
            result = all( 
                equality_check(gt, pred) 
                for gt, pred in  zip(gt_parts, pred_parts) 
            )
    
    return result


print(grade_answer("14/3", r"\frac{14}{3}"))
print(grade_answer(r"(14/3, 2/3)", "(14/3, 4/6)"))

True
True


In [37]:
tests = [
    ("check_1", "3/4", r"\frac{3}{4}", True),
    ("check_2", "(3)/(4)", r"3/4", True),
    ("check_3", r"\frac{\sqrt{8}}{2}", "sqrt(2)", True),
    ("check_4", r"\( \frac{1}{2} + \frac{1}{6} \)", "2/3", True),
    ("check_5", "(1, 2)", r"(1,2)", True),
    ("check_6", "(2, 1)", "(1, 2)", False),
    ("check_7", "(1, 2, 3)", "(1, 2)", False),
    ("check_8", "0.5", "1/2", True),
    ("check_9", "0.3333333333", "1/3", False),
    ("check_10", "1,234/2", "617", True),
    ("check_11", r"\text{2/3}", "2/3", True),
    ("check_12", "50%", "1/2", False),
    ("check_13", r"2\cdot 3/4", "3/2", True),
    ("check_14", r"90^\circ", "90", True),
    ("check_15", r"\left(\frac{3}{4}\right)", "3/4", True),
    ("check_16", r"2²", "2**2", True),
]


def run_demos_table(tests):
    header = ("Test", "Expect", "Got", "Status")
    rows = []
    for name, pred, gtruth, expect in tests:
        got = grade_answer(pred, gtruth)
        status = "PASS" if got == expect else "FAIL"
        rows.append((name, str(expect), str(got), status))

    data = [header] + rows

    col_widths = [
        max(len(row[i]) for row in data)
        for i in range(len(header))
    ]

    for row in data:
        line = " | ".join(
            row[i].ljust(col_widths[i])
            for i in range(len(header))
        )
        print(line)
    
    passed = sum(r[3] == "PASS" for r in rows)
    print(f"\nPassed {passed}/{len(rows)}")


run_demos_table(tests)

Test     | Expect | Got   | Status
check_1  | True   | True  | PASS  
check_2  | True   | True  | PASS  
check_3  | True   | True  | PASS  
check_4  | True   | True  | PASS  
check_5  | True   | True  | PASS  
check_6  | False  | False | PASS  
check_7  | False  | False | PASS  
check_8  | True   | True  | PASS  
check_9  | False  | False | PASS  
check_10 | True   | True  | PASS  
check_11 | True   | True  | PASS  
check_12 | False  | False | PASS  
check_13 | True   | True  | PASS  
check_14 | True   | True  | PASS  
check_15 | True   | True  | PASS  
check_16 | True   | True  | PASS  

Passed 16/16


## 3.8 Loading the evaluation dataset

In [38]:
import json
import requests


def load_math500_test(local_path="math500_test.json", save_copy=True):
    local_path = Path(local_path)
    url = (
        "https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/"
        "main/ch03/01_main-chapter-code/math500_test.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()

        if save_copy:  # Saves a local copy
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data


math_data = load_math500_test()
print("Number of entries:", len(math_data))

Number of entries: 500


In [ ]:
pprint(math_data[0])

{'answer': '\\left( 3, \\frac{\\pi}{2} \\right)',
 'level': 2,
 'problem': 'Convert the point $(0,3)$ in rectangular coordinates to polar '
            'coordinates.  Enter your answer in the form $(r,\\theta),$ where '
            '$r > 0$ and $0 \\le \\theta < 2 \\pi.$',
 'solution': 'We have that $r = \\sqrt{0^2 + 3^2} = 3.$  Also, if we draw the '
             'line connecting the origin and $(0,3),$ this line makes an angle '
             'of $\\frac{\\pi}{2}$ with the positive $x$-axis.\n'
             '\n'
             '[asy]\n'
             'unitsize(0.8 cm);\n'
             '\n'
             'draw((-0.5,0)--(3.5,0));\n'
             'draw((0,-0.5)--(0,3.5));\n'
             'draw(arc((0,0),3,0,90),red,Arrow(6));\n'
             '\n'
             'dot((0,3), red);\n'
             'label("$(0,3)$", (0,3), W);\n'
             'dot((3,0), red);\n'
             '[/asy]\n'
             '\n'
             'Therefore, the polar coordinates are $\\boxed{\\left( 3, '
             '\\frac